# EEGNet — Google Colab

A single workflow for setup, GPU verification, and explicitly selected experiments. Existing results are in the repository's `results/` directory; training is optional.

Prepare the private upload locally with `python scripts/package_colab.py`, then select a GPU runtime. The generated ZIP includes local dataset files and must not be published.


## 1. Upload and install

In [ ]:
from pathlib import Path
import io, sys, subprocess, tempfile, zipfile
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one Lab2_colab.zip")
root = Path(tempfile.mkdtemp(prefix="lab2_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in archive.namelist():
        if not (root / name).resolve().is_relative_to(root.resolve()):
            raise ValueError("Unsafe archive entry")
    archive.extractall(root)
PROJECT_ROOT = root
if not (PROJECT_ROOT / "requirements.txt").exists():
    raise ValueError("Use the ZIP produced by scripts/package_colab.py")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)


## 2. GPU and persistent storage

In [ ]:
import torch
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime")
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/Lab2/outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 3. Short GPU verification

Two epochs test training, artifact saving, and checkpoint reload. These are smoke checks, not full experiment results.

In [ ]:
GPU_SMOKE_VERIFIED = False
subprocess.run([
    sys.executable, str(PROJECT_ROOT / "scripts/run_experiments.py"),
    "--configs", str(PROJECT_ROOT / "configs/selected.json"),
    "--epochs", "2", "--device", "cuda", "--output-dir", str(OUTPUT_ROOT / "smoke"),
], cwd=PROJECT_ROOT, check=True)
GPU_SMOKE_VERIFIED = True


## 4. Optional full experiments

Run only when you intentionally want new training. Choose configurations and model seeds explicitly. The fixed split seed remains 42. Completed matching runs in this output directory are reverified and reused. Interrupted incomplete runs restart from epoch 1.

For the baseline, use `configs/baseline.json`; the selected model is `configs/selected.json`. Controlled changes are in `configs/experiments/` and `configs/combinations/`.


In [ ]:
RUN_FULL_EXPERIMENTS = False
CONFIGS = [PROJECT_ROOT / "configs/selected.json"]
MODEL_SEEDS = [42, 43, 44]
if RUN_FULL_EXPERIMENTS:
    if not GPU_SMOKE_VERIFIED:
        raise RuntimeError("Complete GPU verification first")
    subprocess.run([
        sys.executable, str(PROJECT_ROOT / "scripts/run_experiments.py"),
        "--configs", *map(str, CONFIGS), "--seeds", *map(str, MODEL_SEEDS),
        "--device", "cuda", "--output-dir", str(OUTPUT_ROOT / "experiments"),
    ], cwd=PROJECT_ROOT, check=True)
else:
    print("Full training disabled. Reviewed results are already saved in the repository.")


## 5. Optional export

Raw checkpoints remain private on Drive. Download this archive only after intentionally running full experiments.

In [ ]:
import shutil
if RUN_FULL_EXPERIMENTS:
    archive = shutil.make_archive("/content/Lab2_results", "zip", root_dir=OUTPUT_ROOT / "experiments")
    files.download(archive)
